# NeuroScan Agent: end-to-end training on Colab
Runtime → Change runtime type → **GPU**. Everything is saved to Google Drive so it survives disconnects.

In [1]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/neuroscan'
!mkdir -p $DRIVE/data/raw/kaggle $DRIVE/data/raw/figshare $DRIVE/checkpoints $DRIVE/results

Mounted at /content/drive


In [2]:
# Replace with your repo URL
!git clone https://github.com/hannay-sen/NEUROSCAN.git /content/neuroscan
%cd /content/neuroscan
!pip install -q -r requirements.txt && pip install -q -e .

Cloning into '/content/neuroscan'...
remote: Enumerating objects: 68, done.
remote: Counting objects: 100% (68/68), done.
remote: Compressing objects: 100% (56/56), done.
remote: Total 68 (delta 4), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (68/68), 48.74 KiB | 2.44 MiB/s, done.
Resolving deltas: 100% (4/4), done.
/content/neuroscan
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tweepy 4.17.0 requires oauthlib<4,>=3.2.0, but you have oauthlib 4.0.1 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to b

## Link data, checkpoints and results to Drive
Put the Kaggle *Brain Tumor MRI Dataset* folders (`Training/`, `Testing/`) in `MyDrive/neuroscan/data/raw/kaggle/` and the Figshare `.mat` files in `MyDrive/neuroscan/data/raw/figshare/`.

In [3]:
!rm -rf data checkpoints results
!ln -s $DRIVE/data data
!ln -s $DRIVE/checkpoints checkpoints
!ln -s $DRIVE/results results
!ls data/raw/kaggle data/raw/figshare | head

data/raw/figshare:
1000.mat
1001.mat
1002.mat
1003.mat
1004.mat
1005.mat
1006.mat
1007.mat
1008.mat


## (Optional) download Kaggle data with the Kaggle API
Upload `kaggle.json` from your Kaggle account settings first.

In [4]:
# from google.colab import files; files.upload()   # kaggle.json
# !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !kaggle datasets download -d masoudnickparvar/brain-tumor-mri-dataset -p data/raw/kaggle --unzip

## 1. Data

In [ ]:
!python -m neuroscan.data.prepare_figshare
!python -m neuroscan.data.dedup
!python -m neuroscan.data.make_splits

In [ ]:
import pandas as pd
pd.read_csv('data/splits/dedup_pairs.csv').sample(10, random_state=0)   # spot-check matches

In [ ]:
%cd /content/neuroscan
!rm data && cp -r $DRIVE/data data
!du -sh data

## 2. Train

In [ ]:
!python -m neuroscan.train_classifier
!python -m neuroscan.train_classifier --balance none --tag nobal

In [ ]:
!python -m neuroscan.train_segmentation

## 3. Evaluate

In [ ]:
!python -m neuroscan.eval.evaluate_classifier --split kaggle_test
!python -m neuroscan.eval.evaluate_classifier --split figshare_all
!python -m neuroscan.eval.evaluate_classifier --split kaggle_test --tag nobal
!python -m neuroscan.eval.evaluate_segmentation --split figshare_test
!python -m neuroscan.eval.evaluate_safety --split kaggle_val --sweep

Copy the chosen `min_confidence` / `max_entropy` into `configs/config.yaml`, then:

In [ ]:
!python -m neuroscan.eval.evaluate_safety --split kaggle_test
!ls results

## 4. Try one scan

In [ ]:
import glob
!python -m neuroscan.graph {glob.glob('data/raw/kaggle/Testing/glioma/*')[0]}